# Prediction Model

## Imports

In [ ]:
import json
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Concatenate, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import f1_score, precision_score, recall_score

## Load Training Data

In [ ]:
data_file = "./data/processed/training_data.json"

with open(data_file, "r", encoding="utf-8") as f:
    training_data = json.load(f)

print(f"Loaded {len(training_data)} training examples")


## Prepare Tokenizers

We create tokenizers for IT skills, soft skills, and designations.

In [ ]:
# -----------------------------
# Normalization helpers
# -----------------------------
def normalize_text(text: str):
    return text.lower().strip()

def normalize_skill(skill: str):
    # convert multi-word skills into single tokens
    # "Machine Learning" -> "machine_learning"
    return normalize_text(skill).replace(" ", "_")


# -----------------------------
# Prepare Tokenizers
# -----------------------------
designation_texts = []

it_texts_for_tokenizer = []
soft_texts_for_tokenizer = []

for example in training_data:
    # Current skills
    curr_it = [normalize_skill(s) for s in example.get("it_skill_categories", []) if s.strip()]
    curr_soft = [normalize_skill(s) for s in example.get("soft_skills", []) if s.strip()]
    desig = normalize_skill(example.get("desired_designation", ""))

    designation_texts.append(desig)

    # Next skills
    next_it = [normalize_skill(s) for s in example.get("next_skill", {}).keys() if s.strip()]
    next_soft = [normalize_skill(s) for s in example.get("next_soft_skill", {}).keys() if s.strip()]

    # Combine current + next for tokenizer
    it_texts_for_tokenizer.append(" ".join(curr_it + next_it))
    soft_texts_for_tokenizer.append(" ".join(curr_soft + next_soft))

# Fit tokenizers
it_tokenizer = Tokenizer(oov_token="<OOV>", filters='!"#$%&()*+,-./:;<=>?@[\\]^`{|}~\t\n')
#it_tokenizer = Tokenizer(oov_token="<OOV>")
it_tokenizer.fit_on_texts(it_texts_for_tokenizer)
NUM_IT_SKILLS = len(it_tokenizer.word_index) + 1

# soft_tokenizer = Tokenizer(oov_token="<OOV>")
soft_tokenizer = Tokenizer(oov_token="<OOV>", filters='!"#$%&()*+,-./:;<=>?@[\\]^`{|}~\t\n')
soft_tokenizer.fit_on_texts(soft_texts_for_tokenizer)
NUM_SOFT_SKILLS = len(soft_tokenizer.word_index) + 1

designation_tokenizer = Tokenizer(oov_token="<OOV>")
designation_tokenizer.fit_on_texts(designation_texts)
NUM_DESIGNATIONS = len(designation_tokenizer.word_index) + 1


# -----------------------------
# Debug info
# -----------------------------
print(f"Sample IT skill texts: {it_texts_for_tokenizer[:3]}")
print(f"Sample Soft skill texts: {soft_texts_for_tokenizer[:100]}")
print(
    f"Vocabulary sizes -> IT skills: {NUM_IT_SKILLS}, "
    f"Soft skills: {NUM_SOFT_SKILLS}, "
    f"Designations: {NUM_DESIGNATIONS}"
)

## Prepare Input Sequences

In [ ]:
# Convert to sequences
it_sequences = it_tokenizer.texts_to_sequences(it_texts_for_tokenizer)
soft_sequences = soft_tokenizer.texts_to_sequences(soft_texts_for_tokenizer)
designation_sequences = designation_tokenizer.texts_to_sequences(designation_texts)

# Compute max lengths
MAX_IT_LEN = max(len(seq) for seq in it_sequences)
MAX_SOFT_LEN = max(len(seq) for seq in soft_sequences)

# Pad sequences
X_it = pad_sequences(it_sequences, maxlen=MAX_IT_LEN, padding='post')
X_soft = pad_sequences(soft_sequences, maxlen=MAX_SOFT_LEN, padding='post')
X_designation = np.array([seq[0]-1 if len(seq) > 0 else 0 for seq in designation_sequences])

print(f"Shapes: IT: {X_it.shape}, Soft: {X_soft.shape}, Designation: {X_designation.shape}")

## Prepare Output Labels

We convert next_skill and next_soft_skill into weighted multi-hot vectors.

In [ ]:
NUM_EXAMPLES = len(training_data)
Y_it = np.zeros((NUM_EXAMPLES, NUM_IT_SKILLS), dtype=np.float32)
Y_soft = np.zeros((NUM_EXAMPLES, NUM_SOFT_SKILLS), dtype=np.float32)

for i, example in enumerate(training_data):
    # IT labels
    for skill, count in example.get("next_skill", {}).items():
        norm_skill = normalize_skill(skill)
        print(it_tokenizer.word_index)
        print(soft_tokenizer.word_index)
        print(norm_skill)
        idx = it_tokenizer.word_index.get(norm_skill)
        if idx:
            Y_it[i, idx] = count

    # Soft labels
    for skill, count in example.get("next_soft_skill", {}).items():
        norm_skill = normalize_skill(skill)
        idx = soft_tokenizer.word_index.get(norm_skill)
        if idx:
            Y_soft[i, idx] = count

print("Y_it sum:", Y_it.sum())   # should now be >0
print("Y_soft sum:", Y_soft.sum())

## Train-Test Split

In [ ]:
X_it_train, X_it_test, \
X_soft_train, X_soft_test, \
X_des_train, X_des_test, \
Y_it_train, Y_it_test, \
Y_soft_train, Y_soft_test = train_test_split(
    X_it,
    X_soft,
    X_designation,
    Y_it,
    Y_soft,
    test_size=0.2,
    random_state=42,
    stratify=X_designation
)

print(f"Train samples: {X_it_train.shape[0]}, Test samples: {X_it_test.shape[0]}")

## Build LSTM Model

In [ ]:
EMB_DIM = 64
LSTM_UNITS = 128

# Inputs
it_input = Input(shape=(MAX_IT_LEN,), name="it_input")
soft_input = Input(shape=(MAX_SOFT_LEN,), name="soft_input")
des_input = Input(shape=(1,), name="designation_input")

# Embeddings
it_emb = Embedding(NUM_IT_SKILLS, EMB_DIM, mask_zero=True)(it_input)
soft_emb = Embedding(NUM_SOFT_SKILLS, EMB_DIM, mask_zero=True)(soft_input)
des_emb = Embedding(NUM_DESIGNATIONS, EMB_DIM)(des_input)
des_flat = Flatten()(des_emb)

# LSTM layers
it_lstm = LSTM(LSTM_UNITS)(it_emb)
soft_lstm = LSTM(LSTM_UNITS)(soft_emb)

# Concatenate LSTM outputs + designation
x = Concatenate()([it_lstm, soft_lstm, des_flat])
x = Dense(256, activation="relu")(x)

# Output layers
it_output = Dense(NUM_IT_SKILLS, activation="sigmoid", name="next_skill")(x)
soft_output = Dense(NUM_SOFT_SKILLS, activation="sigmoid", name="next_soft_skill")(x)

# Model
model = Model(inputs=[it_input, soft_input, des_input], outputs=[it_output, soft_output])
model.compile(
    optimizer=Adam(0.001),
    loss={"next_skill": "binary_crossentropy", "next_soft_skill": "binary_crossentropy"},
    metrics={"next_skill": "accuracy", "next_soft_skill": "accuracy"}
)

model.summary()


## Train the Model

In [ ]:
print(sum(sum(Y_it_train)))
print(sum(sum(X_it_train)))
print(X_des_train)
print(sum(sum(Y_soft_train)))
print(Y_soft_train)
history = model.fit(
    [X_it_train, X_soft_train, X_des_train],
    [Y_it_train, Y_soft_train],
    validation_data=([X_it_test, X_soft_test, X_des_test], [Y_it_test, Y_soft_test]),
    epochs=10,
    batch_size=64
)


## Evaluate the Model

In [ ]:
# Cell 9: Evaluate using F1, Precision, Recall

# Predict on test set
Y_it_pred, Y_soft_pred = model.predict([X_it_test, X_soft_test, X_des_test])

# Binarize predictions
Y_it_pred_bin = (Y_it_pred > 0.5).astype(int)
Y_soft_pred_bin = (Y_soft_pred > 0.5).astype(int)

print(Y_it_pred)

# Ensure test labels are integers too
Y_it_test_bin = (Y_it_test > 0).astype(int)
Y_soft_test_bin = (Y_soft_test > 0).astype(int)

# Compute metrics
f1_it = f1_score(Y_it_test_bin, Y_it_pred_bin, average="micro")
precision_it = precision_score(Y_it_test_bin, Y_it_pred_bin, average="micro")
recall_it = recall_score(Y_it_test_bin, Y_it_pred_bin, average="micro")

f1_soft = f1_score(Y_soft_test_bin, Y_soft_pred_bin, average="micro")
precision_soft = precision_score(Y_soft_test_bin, Y_soft_pred_bin, average="micro")
recall_soft = recall_score(Y_soft_test_bin, Y_soft_pred_bin, average="micro")

print("IT Skill Metrics:")
print(f"F1: {f1_it:.3f}, Precision: {precision_it:.3f}, Recall: {recall_it:.3f}")

print("Soft Skill Metrics:")
print(f"F1: {f1_soft:.3f}, Precision: {precision_soft:.3f}, Recall: {recall_soft:.3f}")


## Save the model

In [ ]:
with open("../model/config.json", "w") as f:
    json.dump({
        "MAX_IT_LEN": MAX_IT_LEN,
        "MAX_SOFT_LEN": MAX_SOFT_LEN
    }, f)

model.save("../model/lstm_model.keras")

import json
with open("../model/it_tokenizer.json", "w") as f:
    json.dump(it_tokenizer.to_json(), f)
with open("../model/soft_tokenizer.json", "w") as f:
    json.dump(soft_tokenizer.to_json(), f)
with open("../model/designation_tokenizer.json", "w") as f:
    json.dump(designation_tokenizer.to_json(), f)
